#[5] Advanced Agent

In [ ]:
!pip install -qU "langchain[openai]"

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")


#1. Runtime & State

##1-1. System Message 동적 정의

###① Context정의

In [ ]:
#dataclass
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str

###② Node-style middleware

before_model (Context의 사용자이름 로깅), after_agent (Context의 사용자이름 로깅)

In [ ]:
from langchain.agents.middleware import before_model

@before_model()
def log_before_model(state, runtime):
    print(f"### before_model : state : \n{state}")
    print(f"### before_model : runtime : \n{runtime}")
    print(f"### before_model : 사용자이름 : {runtime.context.user_name}")
    return None

In [ ]:
from langchain.agents.middleware import after_agent

@after_agent
def log_after_agent(state, runtime) :
    print(f"### after_agent : state : {state}")
    print(f"### after_agent : runtime : {runtime}")
    print(f"### after_agent : 사용자이름 : {runtime.context.user_name}")
    return None

###③ Wrap-style middleware

wrap_model_call (context의 사용자이름을 system mesage 에 주입)

In [ ]:
from langchain.messages import SystemMessage
from langchain.agents.middleware import wrap_model_call

@wrap_model_call()
def inject_user_name(request, handler):
    print(f"### wrap_model : request : \n{request}")
    user_name = request.runtime.context.user_name
    print(f"### wrap_model : 사용자이름 : {user_name}")
    if user_name :
        system_message = f"사용자 이름은 {user_name}입니다"
        new_req = request.override(system_prompt=system_message)
        print(f"### wrap_model : new_req : \n{new_req}")
        return handler(new_req)
    return handler(request)


###④ Agent 생성

In [ ]:
from langchain.agents import create_agent
# from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[log_before_model, inject_user_name, log_after_agent],
    context_schema=Context
)

In [ ]:
agent

###⑤ Agent 실행

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    context=Context(user_name="마이콜")
)

In [ ]:
response

##1-2. 모델 동적 정의

###① Context정의

In [ ]:
@dataclass
class Context:
    user_name: str
    is_member : bool = False

###② Wrap-style middleware

wrap_model_call (사용자의 회원여부에 따라 모델을 지정)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import wrap_model_call

@wrap_model_call()
def dynamic_model_selector(request, handler):
    print(f"### request : \n{request}")
    user_name = request.runtime.context.user_name
    is_member = request.runtime.context.is_member
    if is_member :
        model_name = 'gpt-5'
    else :
        model_name = 'gpt-5-mini'

    print(f"모델명 = {model_name}")

    new_model = init_chat_model(model=model_name)
    new_req = request.override(model=new_model)

    return handler(new_req)

###③ Agent 생성

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [ ]:
agent

###④ Agent 실행

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "LangChain 이 뭐죠?"}]},
    context=Context(user_name="둘리", is_member=True)
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

#2. Middleware

##2-1. Built-in middleware

###① LLM tool emulator

In [ ]:
from langchain.tools import tool
from typing import List, Dict

# 이메일 전송 도구
@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """
    지정한 이메일 주소로 메일을 보내는 도구입니다.

    Args:
        to: 수신자 이메일 주소
        subject: 이메일 제목
        body: 이메일 본문 내용
    """
    return f"✅ 이메일이 성공적으로 전송되었습니다.\n수신자: {to}\n제목: {subject}\n내용: {body[:50]}..."


# 이메일 읽기 도구
@tool
def read_email_tool(limit: int = 3) -> str:
    """
    최근 받은 이메일 3개를 읽는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 조회되었습니다."

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="gpt-5-nano",
    tools=[send_email_tool, read_email_tool],
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"), # 디폴트 모델 : anthropic:claude-3-5-sonnet-latest
    ]
)

In [ ]:
agent

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "최근 온 메일 확인하고 알아서 답장해줘."}]})

In [ ]:
print(response["messages"][-1].content)

In [ ]:
response

###② Todo list

In [ ]:
# 이메일 전송 도구
# 이메일 읽기 도구
# emulator 때 사용한 Tool 참고

In [ ]:
# from langchain.agents import create_agent
from langchain.agents.middleware import LLMToolEmulator, TodoListMiddleware

agent = create_agent(
    model="gpt-5-nano",
    tools=[send_email_tool, read_email_tool],
    middleware=[
        LLMToolEmulator(model="gpt-5-nano", tools=[send_email_tool, read_email_tool]),
        TodoListMiddleware(),
    ],
)

In [ ]:
result = agent.invoke({
    "messages": [
            {"role": "user", "content": "온 메일 다 확인한 뒤 나한테 요약해 보고해. 그 다음 답장 작성해 회신 보내줘. 마지막으로 어떻게 보냈는지 보고하고."}
    ]
})

In [ ]:
print(result["messages"][-1].content)

In [ ]:
result

In [ ]:
from langchain.messages import AIMessage

for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    if isinstance(msg, AIMessage) and msg.tool_calls is not None:
        print(f"--- Tool_calls : \n{msg.tool_calls}")
        # print(msg.tool_calls)
    print(f"--- Content : \n{msg.content}")
    print()

###③ Human-in-the-loop

In [ ]:
# 이메일 전송 도구
# 이메일 읽기 도구
# emulator 때 사용한 Tool 참고

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()  #단기메모리 사용

agent = create_agent(
    model="gpt-5-nano",
    tools=[send_email_tool, read_email_tool],
    checkpointer=checkpointer, # 체크포인터 연결
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"),
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 이메일 보내기 전에 사람의 판단 개입
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                # 이메일 읽기는 자동 승인 (사람 개입 없음)
                "read_email_tool": False,
            }
        ),
    ],
)

이메일을 읽을때는 사람 개입 없음

In [ ]:
prompt = "무슨 메일 왔는지 확인해줘"

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-a"}}
)

In [ ]:
response["messages"][-1].pretty_print()

이메일 보내기 전에 사람의 판단 개입


```
  "send_email_tool": {
      "allowed_decisions": ["approve", "edit", "reject"],
  },
```

              

In [ ]:
prompt = "교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-b"}}
)

In [ ]:
response["messages"][-1].pretty_print()

In [ ]:
prompt = """가장 공손하고 격식있는 버전으로 작성하고, 발송 전에 내게 확인을 받고서 발송해줘.
            - 교수님 이메일 : jumany@email.com
            - 소속 학과 : 컴퓨터 공학과
            - 학번 : 20999901
            - 이름 : 빌게이츠
            - 전화번호 : 010-1234-0000
            - 방문 목적 : 취업 상담
            - 방문 시간대 : 모레 오전 9시~12시 """

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-b"}}
)

In [ ]:
response["messages"][-1].pretty_print()

In [ ]:
approved_response = agent.invoke(
    {"messages": [{"role": "user", "content": "이메일을 확인했어요. 이 이메일을 보내세요."}]},
    {"configurable": {"thread_id": "HIL-b", "decision": "approve"}}
)

In [ ]:
approved_response["messages"][-1].pretty_print()

###④ PII Detection

In [ ]:
@tool
def save_customer_feedback(feedback: str) -> str:
    """고객 피드백을 저장하는 도구"""
    return f"📥 고객 피드백 저장 완료: {feedback}"

In [ ]:
from langchain.agents.middleware import LLMToolEmulator, PIIMiddleware

agent = create_agent(
    model="gpt-5-nano",
    tools=[save_customer_feedback],
    middleware=[
        LLMToolEmulator(model="gpt-5-nano"),
        # 이메일 주소는 전부 마스킹 처리
        PIIMiddleware("email", strategy="redact", apply_to_input=True),

        # 카드번호는 마지막 4자리만 남기고 나머지 마스킹 처리
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # API Key 형태(sk-로 시작하는 32자리)는 감지되면 실행 중단
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block", # 차단 (에러 발생)
            apply_to_input=True,
        ),
    ],
)

In [ ]:
prompt = "안녕하세요. 저는 둘리(이메일 : user123@example.com)입니다. 어제 아이폰 구매했는데 결제가 잘 됐는지 확인 부탁합니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
prompt = "안녕하세요. 저는 둘리(이메일 : user123@example.com)입니다. 제 카드번호는 1234123443214321 입니다. 결제 문제를 해결해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
prompt = "안녕하세요. API 호출이 오류가 나요. 제 API Key는 sk-12345678901234567890123456789012 예요. 확인해주세요"

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

Custom PII Detector

In [ ]:
phone_number_detector_regex = r"\b(010)[-\s]?(\d{3,4})[-\s]?(\d{4})\b"

# 커스텀 PII 미들웨어 생성
phone_masking_middleware = PIIMiddleware(
    # pii_type: "phone_number" 라는 커스텀 이름 지정
    pii_type = "phone_number",

    # detector: 위에서 만든 정규식 전달
    detector = phone_number_detector_regex,

    # strategy: "mask" (마스킹)
    # 마스킹은 기본적으로 마지막 4자리를 제외하고 마스킹
    strategy = "mask",

    # apply_to_input: 사용자 입력에 적용
    apply_to_input = True,
)

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[save_customer_feedback],
    middleware=[phone_masking_middleware]
)

In [ ]:
prompt = "안녕하세요, 제 핸드폰 번호는 010-1234-5678 입니다. 등록해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [ ]:
for i, msg in enumerate(response["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

###⑤ Summarize Memory

* model : 요약 작업을 수행할 대상 LLM 모델을 지정 (필수)
* trigger : 메시지 개수나 토큰 수 등 요약 기능이 실행될 '발동 조건' 설정
* keep : 요약 실행 후에도 문맥 유지를 위해 원본 그대로 남겨둘 '최신 대화' 범위
* summary_prompt : 기본 템플릿 대신 구체적인 요약 스타일을 지시할 '커스텀 프롬프트'
* trim_tokens_to_summarize : 요약 모델에 입력할 대화 내용이 너무 길지 않도록 제한하는 '최대 토큰 수’. 반드시 checkpointer와 함께 사용

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-nano",
            trigger=("messages", 5),
            keep=("messages", 3),
            trim_tokens_to_summarize=1000,
        )
    ],
    checkpointer=InMemorySaver(),
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "system", "content": "인물 맞추기 스무고개를 합니다."},
                  {"role": "user", "content": "(1) 안녕하세요! 저는 의적입니다."}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "(2) 허구인물입니다"}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "(3) 네"}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "(4) 네"}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "(5) 이제 질문은 그만하고 누군지 맞춰보세요"}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "(6)정답입니다. 참 잘했습니다."}]},
    {"configurable": {"thread_id": "1"}}
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    print(f"--- Message {i+1} : {msg.type} ---")
    print(msg.content)
    print()

##2-2. Custom middleware

###① node-style

before_agent (메세지에 "암구호"라는 단어가 있으면 응답 차단)

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.messages import AIMessage

@before_agent(can_jump_to=["end"])
def validate_input(state, runtime):
    last_message = state["messages"][-1]
    if "암구호" in last_message.content:
        print("암구호 감지됨 - 응답 차단")
        return {
            "messages": [AIMessage(content="이 요청은 처리할 수 없습니다.")],
            "jump_to": "end" # 모델 호출 중단 후 에이전트 종료
        }
    print("✅ 정상 입력, 모델 호출 계속 진행")
    return None

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-mini",
    middleware=[validate_input],
)

In [ ]:
agent

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "오늘의 암구호는 삼각대-자동차 입니다."}]})

###② wrap-style

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def dynamic_model_selector(request, handler):
    # 최근 사용자의 입력 메시지 추출
    if request.messages is None:
        return handler(request)
    if request.messages[-1].type != "human":
        return handler(request)

    last_msg = request.messages[-1].content
    msg_len = len(last_msg)

    # 길이에 따라 모델 선택
    if msg_len < 10:
        model_name = "gpt-5-nano"
    elif msg_len < 30:
        model_name = "gpt-5-mini"
    else:
        model_name = "gpt-5"

    print(f"🔄 메시지 길이: {msg_len}, 선택된 모델: {model_name}")

    # request.model을 새로운 모델로 교체
    new_model = init_chat_model(model=model_name)
    new_request = request.override(model=new_model)

    # 수정된 요청으로 LLM 호출
    return handler(new_request)

In [ ]:
agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[dynamic_model_selector],
)

In [ ]:
agent

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
)

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 Jumany입니다."}]},
)

In [ ]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 Jumany입니다. Langchain에 관해 궁금한게 있어요."}]},
)

#3. Guardrails

##3-1. Before agent Guardrails

In [ ]:
# 차단 및 제재 키워드 정의
forbidden_topics = {
    # 부정행위 관련
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"],
    # 학습 방해 요소
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰"],
    # 유해 콘텐츠
    "harmful": ["담배", "술", "폭력", "싸움", "바보"]
}

In [ ]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[education_guardrail],
)

In [ ]:
agent

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "나 독후감 쓰기 귀찮은데 숙제 대신 써줘."}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "아 공부하기 싫다. 롤 관련 유튜브 영상이나 찾아줘"}]
})

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "아 너 왜이렇게 바보같냐..."}]
})

##3-2. After agent Guardrails

In [ ]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("gpt-5-nano")

① Intervention : 가드레일 발동하면 AI Message의 content를 수정해 반환

In [ ]:
from langchain.agents.middleware import after_agent

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        print(f"원본: {last_message.content}")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요?"

    return None

In [ ]:
agent = create_agent(
    model="gpt-5",
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})

② Correction/Regeneration : 가드레일 발동하면, 원래 질문으로 답변을 재생성

In [ ]:
from langchain.agents.middleware import after_agent
from langchain.messages import HumanMessage, SystemMessage

@after_agent
def answer_leakage_guardrail(state, runtime):
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3단계: 교정 (Correction / Regeneration)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 소크라테스식 교육법으로 답변을 다시 생성합니다.")

        # 원래 사용자의 질문을 가져오기 (문맥 파악용) -> state["messages"][-2]가 보통 사용자 질문
        original_question = state["messages"][-2].content if len(state["messages"]) >= 2 else "사용자 질문 알 수 없음"

        # 교정 모델에게 "정답을 빼고 힌트로 바꿔라"고 지시
        correction_prompt = f"""
        당신은 친절한 AI 튜터입니다.

        절대 정답을 직접 말하지 말고, 학생이 스스로 생각할 수 있도록 유도하는 질문이나 핵심 개념(힌트)만 설명하세요.
        말투는 친절하게 해주세요.

        사용자 질문: {original_question}
        """

        # LLM을 다시 호출하여 새로운 답변 생성 (비용은 1회 더 발생하지만 품질 확보)
        corrected_response = safety_model.invoke(
            [SystemMessage(content="당신은 소크라테스식 교육법을 사용하는 튜터입니다."),
             HumanMessage(content=correction_prompt)]
        )
        print(f"원본: {last_message.content}")
        # 원래의 유출된 답변을 교정된 답변으로 덮어쓰기
        last_message.content = corrected_response.content

    return None


In [ ]:
agent = create_agent(
    model="gpt-5",
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [ ]:
agent

In [ ]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})

##3-3. Combine multiple Guardrails

before_agent
1. student_safety_middleware : 입력 메세지에서 전화번호/이메일 감지 & 마스킹 처리
2. counseling_escalation_middleware : 심각한 고민/위기 상황 감지 & 상담 이관


In [ ]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [ ]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None


In [ ]:
# 4중 방어막이 적용된 에이전트
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위) - before_agent
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹) - before_agent
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션) - before_agent
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정) - after_agent
    ],
)

In [ ]:
agent

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "저 수학 과외 구하고 싶어요. 제 번호 010-1234-5678로 연락 주세요."}]})

In [ ]:
agent.invoke({"messages": [{"role": "user", "content": "나 요즘 학교에서 왕따 당하는 것 같아서 너무 우울해."}]})

#4. Long-term Memory

##4-1. Use Case 1. 하드코딩으로 Store 저장

① store 생성

     ①-1 InMemoryStore() 객체 생성
     ①-2 store key 목록 정의

In [ ]:
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from collections import defaultdict # To store managed keys per user/app

# Initialize the store
store = InMemoryStore()

# List to keep track of our keys
managed_keys = []
namespace_keys = defaultdict(list)

② store key 정의 - namespace prefix 정의

In [ ]:
user_id = "user_001"
app_context = "personal_assistant"

# Define a namespace for the user and application
namespace_prefix = f"{user_id}::{app_context}::"

③ store data 저장

     ③-1 store_key 정의 : 하드코딩
     ③-2 store_value 정의 : 하드코딩
     ③-3 store 저장 (key, value)
     ③-4 store_key 저장


In [ ]:
# Document 1
key_1 = f"{namespace_prefix}item_001"
doc_1 = Document(
    page_content="사용자는 게임을 좋아함",
    metadata={
        "context": "preferences",
        "source": "chat_history"
    }
)
store.mset([(key_1, doc_1)])
managed_keys.append(key_1)
namespace_keys[(user_id, app_context)].append(key_1)

# Document 2
key_2 = f"{namespace_prefix}item_002"
doc_2 = Document(
    page_content="사용자는 주말에 코딩을 함",
    metadata={
        "context": "activity",
        "source": "user_input"
    }
)
store.mset([(key_2, doc_2)])
managed_keys.append(key_2)
namespace_keys[(user_id, app_context)].append(key_2)

# Document 3
key_3 = f"{namespace_prefix}item_003"
doc_3 = Document(
    page_content="사용자는 초콜릿 아이스크림을 선호함",
    metadata={
        "context": "preferences",
        "source": "user_survey"
    }
)
store.mset([(key_3, doc_3)])
managed_keys.append(key_3)
namespace_keys[(user_id, app_context)].append(key_3)

print(f"Managed keys: {managed_keys}")
print(f"Managed keys map for {(user_id, app_context)}: {namespace_keys[(user_id, app_context)]}")

④ store data 조회

In [ ]:
# Retrieve all documents using the managed_keys list
all_retrieved_documents = store.mget(managed_keys)

print("\nAll retrieved documents:")
for i, doc in enumerate(all_retrieved_documents):
    print(f"\nDocument {i+1} : \nKey : {managed_keys[i]}")
    print(f"Content: {doc.page_content}\nMetadata: {doc.metadata}")


⑤ Context 정의 : user, app 정보 입력

In [ ]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

⑥ @wrap_model_call 정의 - 장기 메모리 데이터 -> 모델 전달

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import SystemMessage
from langchain_core.documents import Document

@wrap_model_call
def inject_memory(request, handler):
    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name

    # namespace_keys에서 현재 사용자/앱에 해당하는 키 목록을 가져옵니다.
    # defaultdict를 사용했으므로 키가 없으면 빈 리스트를 반환합니다.
    keys_to_retrieve = namespace_keys[(current_user, current_app)]
    print(f"키 목록: {keys_to_retrieve}")

    memory_content = "기록된 사용자 특정 정보 없음"
    extracted_facts = []

    if keys_to_retrieve:
        # mget을 사용하여 해당 키에 대한 문서들을 검색합니다.
        retrieved_documents: list[Document | None] = request.runtime.store.mget(keys_to_retrieve)

        for doc in retrieved_documents:
            if doc is not None and isinstance(doc, Document):
                extracted_facts.append(doc.page_content)

    if extracted_facts:
        memory_content = "\n- ".join(extracted_facts)

    system_message = f"\n사용자 관련 장기 메모리 : \n- {memory_content}"
    new_request = request.override(system_prompt=system_message)
    print(f"사용자 {current_user}를 위한 시스템 메시지: {system_message}")
    return handler(new_request)

⑦ agent 생성

      - 모델, store, context_schema
      - middleware 장착

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    store=store,   # store 연결
    context_schema=Context,
    middleware=[inject_memory]  # 미들웨어 장착
)

In [ ]:
agent

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고있는 정보 알려줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

In [ ]:
response["messages"][-1].pretty_print()

##4-2. Use Case 2. uuid

In [ ]:
from langchain_core.stores import InMemoryStore
from collections import defaultdict # To store managed keys per user/app

# Store 초기화
store = InMemoryStore()
# Global dictionary to store managed keys for each user/app namespace
# Each key will be (user_id, app_name) and value will be a list of string keys
managed_keys_map = defaultdict(list)

In [ ]:
from dataclasses import dataclass
from typing import TypedDict, Annotated

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

# LLM이 추출해야 할 정보의 구조를 정의
class UserInfo(TypedDict):
    """사용자의 개인 정보, 민감 정보 및 선호도를 담는 구조체.
    모든 필드는 사용자의 입력 문맥에서 정확히 추출되어야 합니다."""

    user_name: Annotated[str, "The user's real name or nickname. Example: '홍길동', 'Alice'"]
    age: Annotated[int, "The user's age in years. Must be a positive integer. Example: 29"]
    email: Annotated[str, "The user's valid email address. Must follow 'username@domain.com' format. If not present, leave as empty string."]
    preference: Annotated[str, "The user's favorite items or hobbies. Example: '아메리카노', '액션 영화', '재즈 음악'"]
    life_style: Annotated[str, "The user's daily habits, routines or lifestyle description. Example: '아침 조깅', '야간 코딩', '주말 캠핑'"]
    religion: Annotated[str, "The user's religion. Strict examples: '기독교', '천주교', '불교', '무교'. If unknown, set to '무교'"]
    is_premium_member: Annotated[bool, "Boolean flag indicating whether the user is a premium member. Set to True only if explicitly mentioned, otherwise False."]

In [ ]:
# from langchain_core.runnables import RunnableConfig
from langchain.tools import tool
from langchain_core.documents import Document

# Tool 정의: 사용자의 정보를 조회
@tool
def get_user_info(runtime) -> str:
    """
    현재 사용자의 정보 조회 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store
    global managed_keys_map # Access the global managed_keys_map

    # Get all keys for the current user/app from our managed_keys_map
    keys_to_retrieve = managed_keys_map[(user_id, app)]

    if not keys_to_retrieve:
        return "기록된 정보 없음"

    # Use mget to retrieve the documents
    retrieved_documents: list[Document | None] = store.mget(keys_to_retrieve)

    results = []
    for doc in retrieved_documents:
        if doc is not None:
            results.append(doc.page_content)
            for k, v in doc.metadata.items():
                if v:
                    results.append(f"  {k}: {v}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [ ]:
import uuid

@tool
def save_user_info(user_info: UserInfo, runtime):
    """
    사용자의 정보를 저장하거나 업데이트
    사용자 정보가 하나라도 있으면 저장하고, 없으면 저장하지 않음
    """
    user_id = runtime.context.user_id   # 실행 컨텍스트에서 user_id 가져오기
    app = runtime.context.app_name
    store = runtime.store
    global managed_keys_map # Access the global managed_keys_map

    memory_item_id = str(uuid.uuid4())   # Generate a unique key for the memory item
    store_key = f"{user_id}::{app}::{memory_item_id}"    # Using a consistent key format for InMemoryStore

    # Convert user_info (TypedDict) to a Document object
    page_content_parts = []
    metadata = {}
    result = "변경된 정보가 없습니다."
    for u_key, u_value in user_info.items():
        if u_key in ["user_name", "preference", "life_style", "religion"]:
            if u_value: page_content_parts.append(f"{u_key} : {u_value}")
        else:
            if u_value: metadata[u_key] = u_value

    if page_content_parts or metadata:
        document = Document(
            page_content="; ".join(page_content_parts),
            metadata=metadata # Store metadata
        )

        store.mset([(store_key, document)])  # Use mset to store the Document
        managed_keys_map[(user_id, app)].append(store_key)   # Add the key to our managed list for this user/app namespace
        result = f"정보가 저장되었습니다. (ID: {store_key})"

    return result

In [ ]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def prompt_update(request, handler):

    system_message = request.system_prompt or ""
    system_message += """사용자 프롬프트에서 사용자 정보가 인지되면, 정보를 저장해줘.
                         사용자 정보를 묻는 경우, 장기 메모리에 저장된 내용들을 종합해서 친절하게 답변해줘."""

    request = request.override(system_prompt=system_message)
    return handler(request)

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_user_info, save_user_info],
    store=store, # 에이전트에 store 연결
    middleware=[prompt_update], # 미들웨어 장착
    context_schema=Context
)

In [ ]:
agent

In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user",
                   "content": """안녕하세요. 지우입니다.
                                 아침에 일어나서 성경 말씀을 묵상해요. 그 시간이 제겐 힐링이 되는 시간이예요.
                                 말씀 묵상할때 들으면 좋을 음악목록을 제 이메일 aaa@email.com 으로 보내줘요.
                                 40대 여성이며, 독실한 기독교인이예요.
                                 나에 대해 저장한 내용을 알려줘요.
    """}]},
    context=Context(user_id="user_008", app_name="personal_assistant")
)

In [ ]:
response["messages"][-1].pretty_print()

In [ ]:
# 대화형 질의
print("질문을 입력하세요 (종료: exit or quit)")
while True:
    question = input("\n질문: ")
    if question.lower() in ['exit', 'quit', '종료', '끝']:
        print("종료합니다.")
        break

    response = agent.invoke({"messages": [{"role": "user", "content": question}]},
                            context=Context(user_id="user_001", app_name="personal_assistant"))

    # print(f"답변: {response}")
    print(f"답변: {response["messages"][-1].content}")

In [ ]:
# Get all keys for the current user/app from our managed_keys_map
keys_to_retrieve = managed_keys_map[("user_008", "personal_assistant")]

all_docs = store.mget(keys_to_retrieve)
print("\nAll retrieved documents:")
for i, doc in enumerate(all_docs):
    print(f"\nDocument {i+1} : \nKey : {keys_to_retrieve[i]}")
    print(f"Content: {doc.page_content}\nMetadata: {doc.metadata}")


○ Tool을 이용한 장기 메모리 관리 설명

이 섹션에서는 `InMemoryStore`를 사용하여 사용자의 장기 메모리를 관리하는 `get_user_info` 및 `save_user_info` 도구의 작동 방식을 설명합니다.

1.  **`store` (InMemoryStore)**: 모든 사용자 정보를 저장하는 인메모리 저장소입니다. 이 저장소는 키-값 쌍으로 `Document` 객체를 저장합니다.

2.  **`managed_keys_map` (defaultdict)**: 사용자 ID와 애플리케이션 이름의 튜플을 키로 사용하고, 해당 사용자 및 애플리케이션에 속하는 모든 `store_key` 목록을 값으로 저장하는 전역 딕셔너리입니다. 이는 특정 사용자의 모든 저장된 메모리를 추적하는 데 사용됩니다.

○ `save_user_info` 도구의 작동 방식

-   **목적**: `UserInfo` TypedDict 형식으로 제공된 사용자 정보를 `Document` 객체로 변환하여 `InMemoryStore`에 저장합니다.
-   **키 생성**: `user_id`, `app_name`, 그리고 `uuid.uuid4()`를 사용하여 고유한 `store_key`를 생성합니다. 이 키는 `{user_id}::{app_name}::{memory_item_id}` 형식입니다.
-   **`Document` 변환**: `UserInfo` 객체의 필드(예: `user_name`, `age`, `preference` 등)를 `Document`의 `page_content`와 `metadata`로 변환합니다. `page_content`에는 주요 정보가 `성명 : 길동; 선호 항목: 차를 좋아해`와 같이 문자열로 합쳐져 저장되고, 다른 필드는 `metadata`에 저장될 수 있습니다.
-   **저장**: 생성된 `Document`는 `store.mset([(store_key, document)])`를 통해 `InMemoryStore`에 저장됩니다.
-   **키 관리**: 생성된 `store_key`는 `managed_keys_map[(user_id, app)].append(store_key)`를 통해 `managed_keys_map`에 추가되어 나중에 해당 사용자의 모든 정보를 쉽게 조회할 수 있도록 합니다.

○ `get_user_info` 도구의 작동 방식

-   **목적**: 현재 `user_id`와 `app_name`에 해당하는 저장된 모든 사용자 정보를 `InMemoryStore`에서 조회하여 반환합니다.
-   **키 조회**: `managed_keys_map[(user_id, app)]`에서 현재 사용자 및 애플리케이션과 관련된 모든 `store_key` 목록을 가져옵니다.
-   **정보 검색**: `store.mget(keys_to_retrieve)`를 사용하여 이 키 목록에 해당하는 모든 `Document` 객체를 `InMemoryStore`에서 검색합니다.
-   **결과 반환**: 검색된 각 `Document` 객체에서 `page_content`를 추출하여 `results` 리스트에 추가합니다. `metadata`의 추가 정보도 필요에 따라 결과에 포함될 수 있습니다. 최종적으로 이 정보들을 줄바꿈으로 연결된 문자열 형태로 반환합니다.

이 두 도구를 통해 사용자는 구조화된 형식으로 정보를 저장하고, 나중에 해당 정보를 다시 검색하여 에이전트가 사용자 경험을 개인화하거나 특정 작업을 수행하는 데 활용할 수 있습니다.